In [2]:
import pandas as pd
import numpy as np
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Paths
# In Colab, __file__ is not defined. We'll set paths relative to the /content directory.
BASE = "/content" # Assuming the data and output directories are within /content

DATA_DIR   = BASE # master_disaster_data.csv is directly in /content
OUT_DIR    = os.path.join(BASE, "outputs")
MODEL_DIR  = os.path.join(BASE, "models")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True) # Ensure model directory is also created

# 1. Load & Clean
df = pd.read_csv(os.path.join(DATA_DIR, "master_disaster_data.csv"))

# Fix corrupted income row (single extreme negative value)
df = df[df['median_income'] > 0].copy()

# 2. Feature Engineering
print("Engineering features...")

# Per-capita & normalised ratios
df['assistance_per_capita']   = df['total_assistance_amount'] / df['total_population']
df['assistance_per_applicant']= df['total_assistance_amount'] / df['total_applicants'].replace(0, np.nan)
df['applicants_per_capita']   = df['total_applicants'] / df['total_population']
df['housing_share']           = df['housing_assistance_amount'] / df['total_assistance_amount'].replace(0, np.nan)
df['housing_share']           = df['housing_share'].fillna(0).clip(0, 1)

# Socioeconomic risk index (z-score composite)
#  higher poverty_rate → higher risk; higher median_income → lower risk
df['z_poverty']  =  (df['poverty_rate']  - df['poverty_rate'].mean())  / df['poverty_rate'].std()
df['z_income']   = -(df['median_income'] - df['median_income'].mean()) / df['median_income'].std()
df['vulnerability_index'] = (df['z_poverty'] + df['z_income']) / 2

# Income brackets
df['income_bracket'] = pd.cut(df['median_income'],
    bins=[0, 35000, 50000, 65000, 85000, 200000],
    labels=['Very Low', 'Low', 'Middle', 'Upper-Middle', 'High'])

# Poverty tiers
df['poverty_tier'] = pd.cut(df['poverty_rate'],
    bins=[0, 10, 20, 30, 100],
    labels=['Low (<10%)', 'Moderate (10-20%)', 'High (20-30%)', 'Extreme (30%+)'])

# Log transforms (target is right-skewed)
df['log_total_assistance']   = np.log1p(df['total_assistance_amount'])
df['log_assistance_per_cap'] = np.log1p(df['assistance_per_capita'])
df['log_applicants']         = np.log1p(df['total_applicants'])
df['log_population']         = np.log1p(df['total_population'])
df['log_income']             = np.log1p(df['median_income'])

# State-level historical mean (target-encoded feature)
state_mean = df.groupby('state_clean')['log_total_assistance'].mean().rename('state_mean_log_assistance')
df = df.join(state_mean, on='state_clean')

print(f"  Dataset shape after engineering: {df.shape}")
print(f"  New features: assistance_per_capita, vulnerability_index, housing_share, log transforms, state encoding")

# 3. Predictive Modelling: predict log(total_assistance_amount)
FEATURES = [
    'log_population',
    'log_income',
    'poverty_rate',
    'poverty_count',
    'vulnerability_index',
    'applicants_per_capita',
    'log_applicants',
    'housing_share',
    'Year',
    'state_mean_log_assistance',
]
TARGET = 'log_total_assistance'

model_df = df[FEATURES + [TARGET]].dropna()
X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining on {len(X_train)} rows, testing on {len(X_test)} rows")

# Model definitions
models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest":    RandomForestRegressor(n_estimators=200, max_depth=8,
                                              min_samples_leaf=5, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                                    learning_rate=0.05, random_state=42),
}

results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2  = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse= np.sqrt(mean_squared_error(y_test, preds))
    results[name] = {
        "cv_r2_mean": float(cv_scores.mean()),
        "cv_r2_std":  float(cv_scores.std()),
        "test_r2":    float(r2),
        "test_mae":   float(mae),
        "test_rmse":  float(rmse),
        "model":      model,
        "preds":      preds,
    }
    print(f"  {name:25s}  CV R²={cv_scores.mean():.3f}±{cv_scores.std():.3f}  "
          f"Test R²={r2:.3f}  MAE={mae:.3f}  RMSE={rmse:.3f}")

# Best model
best_name = max(results, key=lambda k: results[k]['test_r2'])
best_model = results[best_name]['model']
print(f"\n  ✓ Best model: {best_name}  (Test R²={results[best_name]['test_r2']:.3f})")

# Save best model & metadata
with open(os.path.join(MODEL_DIR, "best_model.pkl"), "wb") as f:
    pickle.dump({"model": best_model, "features": FEATURES, "name": best_name}, f)

metrics_out = {k: {m: v for m, v in v.items() if m != "model" and m != "preds"}
               for k, v in results.items()}
with open(os.path.join(MODEL_DIR, "metrics.json"), "w") as f:
    json.dump(metrics_out, f, indent=2)

# Save engineered dataset
df.to_csv(os.path.join(DATA_DIR, "master_engineered.csv"), index=False)

# 4. Feature Importance Plot
rf_model = results["Random Forest"]["model"]
feat_imp = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#d73027' if v > feat_imp.median() else '#4575b4' for v in feat_imp.values]
feat_imp.plot(kind='barh', ax=ax, color=colors)
ax.set_title("Random Forest – Feature Importance\n(Predicting Total Assistance Amount)", fontsize=13, fontweight='bold')
ax.set_xlabel("Importance Score")
ax.axvline(feat_imp.median(), color='grey', linestyle='--', linewidth=0.8, label='Median')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "feature_importance.png"), dpi=150)
plt.close()

# 5. Actual vs Predicted
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['preds'], alpha=0.4, s=15, color='#2196F3')
    lims = [min(y_test.min(), res['preds'].min()), max(y_test.max(), res['preds'].max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_title(f"{name}\nTest R² = {res['test_r2']:.3f}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Actual log(Assistance)")
    ax.set_ylabel("Predicted log(Assistance)")
    ax.legend(fontsize=8)
plt.suptitle("Actual vs. Predicted – All Models (log scale)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "actual_vs_predicted.png"), dpi=150)
plt.close()

# 6. Model Comparison Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = list(results.keys())
r2s   = [results[n]['test_r2']  for n in names]
maes  = [results[n]['test_mae'] for n in names]
palette = ['#4575b4', '#d73027', '#1a9850']

axes[0].bar(names, r2s, color=palette)
axes[0].set_title("Test R² Score by Model", fontweight='bold')
axes[0].set_ylabel("R² (higher = better)")
axes[0].set_ylim(0, 1)
for i, v in enumerate(r2s):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha='center', fontweight='bold')

axes[1].bar(names, maes, color=palette)
axes[1].set_title("Test MAE by Model\n(log scale units)", fontweight='bold')
axes[1].set_ylabel("MAE (lower = better)")
for i, v in enumerate(maes):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "model_comparison.png"), dpi=150)
plt.close()

# 7. Additional EDA Outputs

# 7a. Vulnerability Index vs Assistance per Capita
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(df['vulnerability_index'], df['log_assistance_per_cap'],
                c=df['Year'], cmap='viridis', alpha=0.5, s=20)
plt.colorbar(sc, ax=ax, label='Year')
ax.set_xlabel("Vulnerability Index\n(Higher = More Socioeconomically Vulnerable)")
ax.set_ylabel("Log(Assistance per Capita)")
ax.set_title("Socioeconomic Vulnerability vs. Per-Capita Disaster Assistance", fontsize=12, fontweight='bold')
# Add trend line
z = np.polyfit(df['vulnerability_index'].dropna(), df.loc[df['vulnerability_index'].notna(), 'log_assistance_per_cap'], 1)
p = np.poly1d(z)
xline = np.linspace(df['vulnerability_index'].min(), df['vulnerability_index'].max(), 100)
ax.plot(xline, p(xline), 'r--', linewidth=2, label=f'Trend line')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "vulnerability_vs_assistance.png"), dpi=150)
plt.close()

# 7b. State-level summary heatmap
state_year = df.groupby(['state_clean', 'Year'])['total_assistance_amount'].sum().unstack(fill_value=0)
state_year_log = np.log1p(state_year)
fig, ax = plt.subplots(figsize=(10, 16))
sns.heatmap(state_year_log, cmap='YlOrRd', annot=False, linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Log(Total Assistance $)'})
ax.set_title("Total FEMA Assistance by State & Year\n(Log Scale)", fontsize=13, fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("State")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "state_year_heatmap.png"), dpi=150)
plt.close()

# 7c. Distribution of assistance_per_capita by income bracket
fig, ax = plt.subplots(figsize=(10, 6))
order = ['Very Low', 'Low', 'Middle', 'Upper-Middle', 'High']
plot_df = df[df['income_bracket'].notna() & df['assistance_per_capita'].notna()]
sns.boxplot(data=plot_df, x='income_bracket', y='log_assistance_per_cap',
            order=order, palette='coolwarm', ax=ax)
ax.set_title("Per-Capita Assistance by County Income Bracket", fontsize=12, fontweight='bold')
ax.set_xlabel("Median Household Income Bracket")
ax.set_ylabel("Log(Assistance per Capita $)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "assistance_by_income_bracket.png"), dpi=150)
plt.close()

# 7d. Housing assistance share by state (top 15)
top_states = df.groupby('state_clean')['total_assistance_amount'].sum().nlargest(15).index
state_housing = df[df['state_clean'].isin(top_states)].groupby('state_clean').agg(
    housing=('housing_assistance_amount','sum'),
    total=('total_assistance_amount','sum')
).assign(housing_pct=lambda x: 100*x['housing']/x['total']).sort_values('total', ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(state_housing))
w = 0.35
ax.bar(x - w/2, state_housing['total']/1e6, w, label='Total Assistance', color='#4575b4')
ax.bar(x + w/2, state_housing['housing']/1e6, w, label='Housing Assistance', color='#d73027')
ax.set_xticks(x); ax.set_xticklabels(state_housing.index)
ax.set_ylabel("Assistance ($ Millions)")
ax.set_title("Total vs. Housing Assistance – Top 15 States (2019–2022)", fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "total_vs_housing_by_state.png"), dpi=150)
plt.close()

print("\n All outputs saved to:", OUT_DIR)
print("Best model saved to:", MODEL_DIR)

Engineering features...
  Dataset shape after engineering: (3610, 27)
  New features: assistance_per_capita, vulnerability_index, housing_share, log transforms, state encoding

Training on 2888 rows, testing on 722 rows
  Ridge Regression           CV R²=0.790±0.024  Test R²=0.829  MAE=0.423  RMSE=0.870
  Random Forest              CV R²=0.853±0.032  Test R²=0.896  MAE=0.329  RMSE=0.678
  Gradient Boosting          CV R²=0.853±0.019  Test R²=0.870  MAE=0.325  RMSE=0.759

  ✓ Best model: Random Forest  (Test R²=0.896)

 All outputs saved to: /content/outputs
Best model saved to: /content/models
